In [24]:
from pathlib import Path
import re
import html
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Models").exists() and (candidate / "Dataset source").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing Models and Dataset source.")

PROJECT_ROOT = find_project_root(Path.cwd())
BASE_DIR = PROJECT_ROOT / "Dataset source" / "training"
file_path = BASE_DIR / "Dataset_new.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig", usecols=[0, 1])
df.columns = ["Category", "Message"]


In [25]:
def clean_message(text):
    s = str(text).strip()

    s = html.unescape(s)

    s = re.sub(r"^\*\s*Am\b", "I am", s, flags=re.I)
    s = re.sub(r"^\*\s+", "", s)
    s = re.sub(r"^[\?\.,;:]+\s*", "", s)

    s = re.sub(r"<\s*#\s*>", "<NUM>", s)
    s = re.sub(r"<\s*url\s*>", "<URL>", s, flags=re.I)

    s = s.replace("0A$", "<CUR>")
    s = re.sub(r"<NUM>\s*%of", "<NUM> % of", s, flags=re.I)

    s = re.sub(r"\s+", " ", s).strip()
    return s


In [26]:
check_df = df.copy()
check_df = check_df.dropna(subset=["Category", "Message"]).copy()
check_df["Category"] = check_df["Category"].astype(str).str.strip().str.lower()
check_df = check_df[check_df["Category"].isin(["ham", "spam"])]
check_df["Message"] = check_df["Message"].apply(clean_message)

conflict_df = (
    check_df.groupby("Message")["Category"]
    .nunique()
    .reset_index(name="label_count")
)
conflict_df = conflict_df[conflict_df["label_count"] > 1]

print(f"Conflicting messages: {len(conflict_df)}")
conflict_df.head()


Conflicting messages: 0


,Message,label_count


In [27]:
df["Category"] = df["Category"].astype(str).str.strip().str.lower()
df = df[df["Category"].isin(["ham", "spam"])]
df["Category"].value_counts()


Category
ham     17467
spam     8483
Name: count, dtype: int64

In [28]:
df = df.dropna(subset=["Category", "Message"]).copy()
df["Category"] = df["Category"].astype(str).str.strip().str.lower()
df = df[df["Category"].isin(["ham", "spam"])]
df["Message"] = df["Message"].apply(clean_message)
df = df[df["Message"] != ""]
df = df.drop_duplicates(keep="first").reset_index(drop=True)

print(df.shape)
df.head()


(24212, 2)


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [29]:
out_path = BASE_DIR / "Dataset_new_clean.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path}")


Saved: c:\Users\LCZ\Desktop\Msc Project\Dataset source\training\Dataset_new_clean.csv
